Các thư viện bổ sung sau đây là cần thiết để chạy notebook này. Lưu ý rằng việc chạy trên Colab vẫn đang trong quá trình thử nghiệm, vui lòng báo cáo sự cố trên Github nếu bạn gặp bất kỳ vấn đề nào.

In [ ]:
# Cài đặt thư viện d2l phiên bản 1.0.3
!pip install d2l==1.0.3

# Mô hình Phân loại Cơ bản
:label:`sec_classification`

Bạn có thể đã nhận ra rằng các triển khai từ đầu (scratch) và triển khai rút gọn sử dụng các chức năng có sẵn của framework là khá tương tự nhau trong trường hợp của bài toán hồi quy. Điều tương tự cũng đúng đối với bài toán phân loại. Vì nhiều mô hình trong cuốn sách này giải quyết bài toán phân loại, việc thêm các chức năng hỗ trợ riêng cho bối cảnh này là rất đáng giá. Phần này cung cấp một lớp cơ sở (base class) cho các mô hình phân loại để đơn giản hóa mã nguồn trong tương lai.

In [ ]:
import torch
from d2l import torch as d2l  # Nhập thư viện d2l sử dụng backend PyTorch

## Lớp `Classifier`

Chúng ta định nghĩa lớp `Classifier` bên dưới. Trong bước đánh giá (`validation_step`), chúng ta báo cáo cả giá trị mất mát (loss) và độ chính xác phân loại (accuracy) trên một lô đánh giá (validation batch). Chúng ta vẽ biểu đồ cập nhật sau mỗi `num_val_batches` lô. Điều này có lợi ích là tạo ra giá trị mất mát và độ chính xác trung bình trên toàn bộ dữ liệu đánh giá. Các con số trung bình này sẽ không hoàn toàn chính xác nếu lô cuối cùng chứa ít mẫu hơn, nhưng chúng ta bỏ qua sự khác biệt nhỏ này để giữ cho mã nguồn đơn giản.

In [ ]:
class Classifier(d2l.Module):  #@save
    """Lớp cơ sở cho các mô hình phân loại."""
    def validation_step(self, batch):
        # Dự đoán đầu ra Y_hat từ các đặc trưng đầu vào (tất cả các phần tử trừ nhãn cuối cùng)
        Y_hat = self(*batch[:-1])
        # Vẽ biểu đồ giá trị loss cho tập validation
        self.plot('loss', self.loss(Y_hat, batch[-1]), train=False)
        # Vẽ biểu đồ độ chính xác (accuracy) cho tập validation
        self.plot('acc', self.accuracy(Y_hat, batch[-1]), train=False)

Theo mặc định, chúng ta sử dụng bộ tối ưu hóa hạ cực gradient ngẫu nhiên (SGD), hoạt động trên các lô nhỏ (minibatches), giống như chúng ta đã làm trong ngữ cảnh của hồi quy tuyến tính.

In [ ]:
@d2l.add_to_class(d2l.Module)  #@save
def configure_optimizers(self):
    # Sử dụng bộ tối ưu hóa SGD (Stochastic Gradient Descent) với tốc độ học lr
    return torch.optim.SGD(self.parameters(), lr=self.lr)

## Độ chính xác (Accuracy)

Khi nhận được phân phối xác suất dự đoán `y_hat`,
chúng ta thường chọn lớp có xác suất dự đoán cao nhất
mỗi khi phải đưa ra một dự đoán cứng (hard prediction).
Thực tế, nhiều ứng dụng yêu cầu chúng ta phải đưa ra lựa chọn.
Ví dụ, Gmail phải phân loại một email vào các mục "Hộp thư chính", "Mạng xã hội", "Cập nhật", "Diễn đàn", hoặc "Thư rác".
Nó có thể ước tính xác suất nội bộ,
nhưng cuối cùng nó vẫn phải chọn một trong số các lớp đó.

Khi các dự đoán khớp với nhãn thực tế `y`, chúng được coi là chính xác.
Độ chính xác phân loại là tỷ lệ của tất cả các dự đoán chính xác.
Mặc dù có thể khó tối ưu hóa trực tiếp độ chính xác (vì nó không khả vi),
nhưng nó thường là thước đo hiệu suất mà chúng ta quan tâm nhất. Nó thường là đại lượng
quan trọng trong các bài kiểm tra đánh giá (benchmarks). Vì thế, chúng ta hầu như luôn báo cáo nó khi huấn luyện các bộ phân loại.

Độ chính xác được tính như sau.
Đầu tiên, nếu `y_hat` là một ma trận,
chúng ta giả định rằng chiều thứ hai lưu trữ điểm số dự đoán cho mỗi lớp.
Chúng ta sử dụng `argmax` để lấy lớp được dự đoán bằng chỉ số có giá trị lớn nhất trong mỗi hàng.
Sau đó, chúng ta [**so sánh lớp được dự đoán với nhãn thực tế `y` theo từng phần tử.**]
Vì toán tử so sánh bằng `==` nhạy cảm với kiểu dữ liệu,
chúng ta chuyển đổi kiểu dữ liệu của `y_hat` để khớp với kiểu dữ liệu của `y`.
Kết quả là một tensor chứa các phần tử có giá trị 0 (sai) và 1 (đúng).
Tính tổng các phần tử này sẽ cho ta số lượng dự đoán chính xác.

In [ ]:
@d2l.add_to_class(Classifier)  #@save
def accuracy(self, Y_hat, Y, averaged=True):
    """Tính toán số lượng dự đoán chính xác."""
    # Định hình lại Y_hat về dạng ma trận 2D
    Y_hat = Y_hat.reshape((-1, Y_hat.shape[-1]))
    # Lấy chỉ số có giá trị lớn nhất theo hàng (dự đoán lớp) và chuyển kiểu dữ liệu cho khớp với Y
    preds = Y_hat.argmax(axis=1).type(Y.dtype)
    # So sánh kết quả dự đoán với nhãn thực tế Y
    compare = (preds == Y.reshape(-1)).type(torch.float32)
    # Trả về độ chính xác trung bình hoặc mảng so sánh chi tiết
    return compare.mean() if averaged else compare

## Tóm tắt

Phân loại là một bài toán rất phổ biến nên nó cần có các hàm tiện ích riêng. Tầm quan trọng cốt lõi trong bài toán phân loại là *độ chính xác* của bộ phân loại. Lưu ý rằng mặc dù chúng ta thường quan tâm chủ yếu đến độ chính xác, chúng ta huấn luyện các bộ phân loại để tối ưu hóa nhiều mục tiêu khác vì lý do thống kê và tính toán. Tuy nhiên, bất kể hàm mất mát nào được giảm thiểu trong quá trình huấn luyện, việc có một phương thức tiện ích để đánh giá độ chính xác của bộ phân loại theo thực nghiệm là rất hữu ích.


## Bài tập

1. Ký hiệu $L_\textrm{v}$ là loss trên tập validation, và gọi $L_\textrm{v}^\textrm{q}$ là ước lượng sơ bộ của nó được tính bằng cách trung bình hóa hàm loss trong phần này. Cuối cùng, ký hiệu $l_\textrm{v}^\textrm{b}$ là loss trên minibatch cuối cùng. Hãy biểu diễn $L_\textrm{v}$ thông qua $L_\textrm{v}^\textrm{q}$, $l_\textrm{v}^\textrm{b}$, kích thước tập mẫu và kích thước minibatch.
1. Chứng minh rằng ước lượng sơ bộ $L_\textrm{v}^\textrm{q}$ là không chệch (unbiased). Nghĩa là, chứng minh rằng $E[L_\textrm{v}] = E[L_\textrm{v}^\textrm{q}]$. Tại sao bạn vẫn muốn sử dụng $L_\textrm{v}$ thay thế?
1. Cho trước một hàm loss phân loại nhiều lớp, ký hiệu $l(y,y')$ là hình phạt khi ước lượng $y'$ trong khi nhãn thực tế là $y$, và cho trước xác suất $p(y \mid x)$, hãy thiết lập quy tắc để lựa chọn tối ưu $y'$. Gợi ý: biểu diễn hàm loss kỳ vọng, sử dụng $l$ và $p(y \mid x)$.

[Thảo luận](https://discuss.d2l.ai/t/6809)